## Notebook 04: Export Data for Power BI
### Volkswagen Stock Data 2000–2025

This notebook creates three structured Excel files that Power BI will import.
Each file serves a different dashboard page.

Output: data/processed/

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\vw_project\data\processed\vw_stock_features.csv',
                 index_col='Date',
                 parse_dates=True)

print('Data loaded:', df.shape)

Data loaded: (6684, 25)


In [7]:
daily = df[['Open','High','Low','Close','Volume',
            'Daily_Return_Pct','MA_50','MA_200',
            'Volatility_30d','RSI_14','MACD',
            'Year','Month','Quarter','Day_of_Week']].copy()

daily.to_excel(r'C:\vw_project\data\processed\powerbi_daily.xlsx')
print(f'Daily table exported: {len(daily):,} rows')

Daily table exported: 6,684 rows


In [8]:
annual = df.groupby('Year').agg(
    Open_Year       = ('Open','first'),
    Close_Year      = ('Close','last'),
    High_Year       = ('High','max'),
    Low_Year        = ('Low','min'),
    Avg_Volume      = ('Volume','mean'),
    Avg_Volatility  = ('Volatility_30d','mean'),
    Avg_RSI         = ('RSI_14','mean')
).reset_index()

annual['Annual_Return_Pct'] = (
    (annual['Close_Year'] - annual['Open_Year']) / annual['Open_Year'] * 100
).round(2)

annual.to_excel(r'C:\vw_project\data\processed\powerbi_annual.xlsx', index=False)
print(f'Annual table exported: {len(annual)} rows')

Annual table exported: 27 rows


In [9]:
monthly = df.groupby(['Year','Month']).agg(
    Avg_Close       = ('Close','mean'),
    Total_Volume    = ('Volume','sum'),
    Avg_Return      = ('Daily_Return_Pct','mean'),
    Avg_RSI         = ('RSI_14','mean'),
    Avg_Volatility  = ('Volatility_30d','mean')
).reset_index()

monthly.to_excel(r'C:\vw_project\data\processed\powerbi_monthly.xlsx', index=False)
print(f'Monthly table exported: {len(monthly)} rows')

Monthly table exported: 314 rows


In [10]:
import os

files = [
    r'C:\vw_project\data\processed\powerbi_daily.xlsx',
    r'C:\vw_project\data\processed\powerbi_annual.xlsx',
    r'C:\vw_project\data\processed\powerbi_monthly.xlsx',
]

for f in files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) / 1024 if exists else 0
    print(f'{"OK" if exists else "MISSING"} — {os.path.basename(f)} ({size:.0f} KB)')

OK — powerbi_daily.xlsx (653 KB)
OK — powerbi_annual.xlsx (7 KB)
OK — powerbi_monthly.xlsx (23 KB)
